# 4.10 · Ноутбук 1. Дані: data card і протокол розподілу
**Проєкт:** класифікація зображень рукописних цифр: простий baseline проти згорткової нейронної мережі (CNN).
**Варіант завдання:** 1, Image classification.

Цей ноутбук: (1) завантажує відкритий набір даних, (2) описує його (data card), (3) фіксує відтворюваний розподіл train / validation / test і зберігає індекси в `data/split.npz`.

In [ ]:
import os, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%config InlineBackend.figure_formats = ["jpeg"]
plt.rcParams["figure.dpi"] = 80
sns.set_theme(style="whitegrid")

CFG = json.load(open("../config.json", encoding="utf-8"))
SEED = CFG["seed"]

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

## 1. Завантаження даних
**Джерело:** UCI ML «Optical Recognition of Handwritten Digits» у версії, що постачається разом із scikit-learn (`sklearn.datasets.load_digits`). Інтернет для завантаження не потрібен, дані відкриті, персональних або службових даних немає.

In [ ]:
digits = load_digits()
X, y = digits.images, digits.target        # X: (n, 8, 8), значення яскравості 0..16
print("Зображень:", X.shape[0], "| розмір:", X.shape[1:], "| класів:", len(np.unique(y)))
print("Діапазон значень пікселів:", X.min(), "–", X.max())
print("Пропусків:", np.isnan(X).sum())

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(14, 4.5))
for ax, i in zip(axes.ravel(), np.random.default_rng(SEED).choice(len(X), 30, replace=False)):
    ax.imshow(X[i], cmap="gray_r"); ax.set_title(int(y[i])); ax.axis("off")
plt.suptitle("Випадкові приклади зображень (8×8 пікселів)"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
counts = np.bincount(y)
axes[0].bar(range(10), counts, color="#457b9d"); axes[0].set_xticks(range(10))
axes[0].set_title("Кількість зображень у класах"); axes[0].set_xlabel("Цифра")
mean_imgs = np.hstack([X[y == k].mean(axis=0) for k in range(10)])
axes[1].imshow(mean_imgs, cmap="gray_r"); axes[1].axis("off"); axes[1].set_title("Середнє зображення кожного класу (0 … 9)")
plt.tight_layout(); plt.show()
print("Мін / макс кількість у класі:", counts.min(), "/", counts.max())

## 2. Data card
| Поле | Значення |
|---|---|
| Джерело / provenance | UCI ML Optical Recognition of Handwritten Digits (Alpaydin & Kaynak, 1998), копія в scikit-learn |
| Ліцензія / чутливість | відкриті дані, без персональної чи службової інформації |
| Обсяг | 1797 зображень, 10 класів (цифри 0–9) |
| Схема | зображення 8×8, яскравість 0–16 (цілі), мітка класу 0–9 |
| Баланс класів | 174–183 зображення на клас, практично збалансовано |
| Обмеження | дуже низька роздільність (8×8), рукопис обмеженої кількості авторів, чисті сканування без шуму |

## 3. Протокол розподілу
Стратифікований розподіл **60 / 20 / 20** з фіксованим seed:
- **train** (≈1078) для навчання моделей;
- **validation** (≈360) для вибору моделі та гіперпараметрів і ранньої зупинки;
- **test** (≈359) лише для однієї фінальної оцінки після завершення tuning.

Індекси зберігаються у файлі, тож усі наступні ноутбуки використовують однаковий розподіл.

**Контроль витоку (leakage):** масштабування обчислюється тільки на train. Test не використовується ні для вибору архітектури, ні для ранньої зупинки. Нижче додатково перевіряємо, що між частинами немає ідентичних зображень.

In [ ]:
idx = np.arange(len(X))
idx_train, idx_tmp = train_test_split(idx, test_size=0.4, stratify=y, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, stratify=y[idx_tmp], random_state=SEED)
os.makedirs("../data", exist_ok=True)
np.savez("../data/split.npz", train=idx_train, val=idx_val, test=idx_test)

print("train / val / test:", len(idx_train), len(idx_val), len(idx_test))
print("Перетин індексів:", len(set(idx_train) & set(idx_val)), len(set(idx_train) & set(idx_test)), len(set(idx_val) & set(idx_test)))

flat = X.reshape(len(X), -1)
h = lambda ids: {flat[i].tobytes() for i in ids}
print("Однакові зображення train∩test:", len(h(idx_train) & h(idx_test)), "| train∩val:", len(h(idx_train) & h(idx_val)))
pd.DataFrame({s: np.bincount(y[i]) for s, i in [("train", idx_train), ("val", idx_val), ("test", idx_test)]}).T